<font size="6">Calling the main libraries</font>

In [2]:
import torch
import torch.nn as nn
from torchvision import transforms , datasets
from torch.utils.data import DataLoader , random_split
import numpy as np
import torch.optim as optim

<font size="6">The model code</font>

In [3]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
        )
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, X):
        x = self.features(X)
        x = self.flatten(x)
        return self.classifier(x)

<font size="6">The training code</font>


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def train(train_loader, val_loader, epochs, model):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            optimizer.zero_grad()
            #loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                val_loss += loss_fn(logits, yb).item()
                correct += (logits.argmax(1) == yb).sum().item()
                total += yb.size(0)

        print(f"Epoch {epoch+1}/{epochs} - train loss: {running_loss/len(train_loader):.4f} "
              f"- val loss: {val_loss/len(val_loader):.4f} - val acc: {correct/total:.3f}")

    return model

<font size="6">Loading the trained model</font>

In [5]:
model1 = CNN(num_classes=120).to(device)
model1.load_state_dict(torch.load("../Model/model1.pth" ,weights_only=False , map_location=torch.device("cpu")))

<All keys matched successfully>

<font size="6">Creating the dataloader then start the training</font>

In [ ]:
from torch.utils.data import random_split

print(torch.cuda.is_available())

tranfrom = transforms.Compose([
    transforms.Resize((244, 244)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(root='../assets/one example of each dog', transform=tranfrom)
num_classes = len(dataset.classes)
print("num_classes:", num_classes)

n_val = int(0.05 * len(dataset))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0, pin_memory=False) # Changed pin_memory to False
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=False) # Changed pin_memory to False
model1 = train(train_loader, val_loader, epochs=30, model=model1)

<font size="6">Saving the model</font>

In [ ]:
torch.save(model1.state_dict() , "/content/drive/MyDrive/model1.pth")